[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/ai-agents-certified/notebooks/day-10-capstone-research-system.ipynb#scrollTo=ca1b2c3d)

---
# Day 10 · Capstone — Multi-Agent Research System with Persistence and Streaming
**certified-journeys / ai-agents-certified** · Day 10 · Capstone

> **Goal for today:** Build a production-grade multi-agent research system — supervisor node, research subagent with DuckDuckGo search, summarizer subagent, SQLite state persistence, node-level streaming to the terminal, and a structured Markdown research report as the final output.


## Capstone Architecture

Before writing any code, read this architecture diagram carefully. The graph structure is the hardest thing to change later — get it right on paper first.

```
Human input
     │
     ▼
┌──────────────────────────────────────────────────────────┐
│                    ResearchState                         │
│  messages       (shared, append-only)                    │
│  next           (supervisor routing)                     │
│  scratchpad     (researcher-owned: intermediate notes)   │
│  report         (summarizer-owned: final Markdown)       │
│  retry_count    (loop guard)                             │
└──────────────────────────────────────────────────────────┘
     │
     ▼
[supervisor] ──────────────────────────────────────────────
     │ "researcher"          │ "summarizer"    │ "FINISH"
     ▼                       ▼                 ▼
[researcher]            [summarizer]          END
 - DuckDuckGo search    - reads scratchpad
 - writes scratchpad    - writes report
     │                       │
     └───────────────────────┘
           back to supervisor

SQLite checkpointer: persists ResearchState across invocations
Streaming: node labels (updates) + tokens (messages)
```

**Decision: Why SQLite over MemorySaver?**
SQLite persists between Python process restarts. This means:
- If the research run is interrupted, it resumes from the last checkpoint.
- Multiple research questions in the same `thread_id` accumulate context.
- The final report can be retrieved hours or days later.


In [ ]:
%pip install -q langgraph langchain-core langchain-openai langgraph-checkpoint-sqlite duckduckgo-search pydantic


## Step 1 · State Schema and SQLite Checkpointer

Set up persistence **before** writing any agent logic. This lets you verify state survives across invocations immediately, without having to debug the full graph later.

**Verification protocol:**
1. Compile graph with SQLite checkpointer.
2. Invoke with a simple initial state.
3. Kill the process (or just re-invoke with the same `thread_id`).
4. Confirm the state from step 2 is recovered.


In [ ]:
import os, asyncio
from typing import Annotated, Literal, Optional, TypedDict
from pydantic import BaseModel
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.runnables import RunnableConfig
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.sqlite import SqliteSaver

os.environ.setdefault("OPENAI_API_KEY", "sk-placeholder")  # replace with real key

# ── State schema ──────────────────────────────────────────────────────────────
class ResearchState(TypedDict):
    messages:     Annotated[list, add_messages]  # full conversation thread
    next:         str                             # supervisor routing field
    scratchpad:   Optional[str]                   # researcher's working notes
    report:       Optional[str]                   # final Markdown report
    retry_count:  int                             # loop break guard

# ── SQLite checkpointer ───────────────────────────────────────────────────────
DB_PATH = "/tmp/research_system.db"
checkpointer = SqliteSaver.from_conn_string(DB_PATH)

print("State schema defined:")
for field, annotation in ResearchState.__annotations__.items():
    print(f"  {field}: {annotation}")

print(f"\nSQLite checkpointer initialized: {DB_PATH}")
print("State will persist across Python process restarts.")


**What just happened?**
- **`SqliteSaver.from_conn_string(path)`** creates (or opens) a SQLite database at the given path. The schema is managed by LangGraph — you never write SQL directly.
- We defined `ResearchState` with five fields: the shared message thread, the routing field, two private namespace fields, and a loop guard counter.
- Persistence is established **before** any agent logic — this is the verification-first approach that prevents hard-to-debug issues later.


## Step 2 · Persistence Verification

Before writing agent nodes, verify that the checkpointer actually saves and restores state. A minimal stub graph is enough to test this.


In [ ]:
# Build a stub graph just to verify SQLite persistence
def stub_node(state: ResearchState) -> dict:
    """Does nothing except confirm the node ran."""
    return {
        "messages": [AIMessage(content="stub ran")],
        "next": "FINISH",
    }

stub_builder = StateGraph(ResearchState)
stub_builder.add_node("stub", stub_node)
stub_builder.set_entry_point("stub")
stub_builder.add_edge("stub", END)
stub_graph = stub_builder.compile(checkpointer=checkpointer)

THREAD_ID = "capstone-verify-001"
config = {"configurable": {"thread_id": THREAD_ID}}

# First invocation — runs the stub node
result = stub_graph.invoke(
    {"messages": [HumanMessage(content="verify persistence")],
     "next": "", "scratchpad": None, "report": None, "retry_count": 0},
    config=config
)
print("First invocation complete.")
print(f"  messages count: {len(result['messages'])}")
print(f"  last message: {result['messages'][-1].content}")

# Retrieve the persisted state
snapshot = stub_graph.get_state(config)
print(f"\nPersisted state retrieved from SQLite:")
print(f"  messages count: {len(snapshot.values['messages'])}")
print(f"  last message: {snapshot.values['messages'][-1].content}")
print(f"  next: '{snapshot.values['next']}'")
print(f"  scratchpad: {snapshot.values['scratchpad']}")
print(f"  report: {snapshot.values['report']}")
print("\n✓ SQLite persistence verified — safe to build agent logic")


**What just happened?**
- **`graph.get_state(config)`** retrieves the latest checkpoint for a given `thread_id` without re-running the graph.
- The stub graph confirms the entire persistence path — compile → invoke → checkpoint write → get_state → verify — with zero real LLM calls.
- If this step fails (file permissions, missing package, etc.), you catch it now rather than after writing 200 lines of agent code.


## Step 3 · Supervisor Router

The supervisor classifies the current research task and routes to the appropriate subagent. Its system prompt must describe exactly when to route to each agent and when to FINISH.


In [ ]:
from unittest.mock import MagicMock

class RouteDecision(BaseModel):
    next: Literal["researcher", "summarizer", "FINISH"]

SUPERVISOR_SYSTEM = """\
You are a research supervisor coordinating a team of two agents:
  - researcher: searches the web for facts, data, and sources
  - summarizer: condenses the research scratchpad into a structured Markdown report

Routing rules:
1. If the scratchpad is empty or insufficient, route to 'researcher'.
2. If the scratchpad has enough data for a good report, route to 'summarizer'.
3. If the report is complete (non-empty), output 'FINISH'.
4. If retry_count >= 2 and scratchpad is still empty, output 'FINISH' to break the loop.

Respond with exactly one word from: researcher, summarizer, FINISH."""

def make_supervisor(llm):
    """Return a supervisor node that routes using structured output."""
    router = llm.with_structured_output(RouteDecision)

    def supervisor_node(state: ResearchState) -> dict:
        # Build context for the router: system + current state summary + messages
        context = f"\n\nCurrent state: scratchpad={'[set]' if state.get('scratchpad') else '[empty]'}, " \
                  f"report={'[set]' if state.get('report') else '[empty]'}, " \
                  f"retry_count={state.get('retry_count', 0)}"
        system = SystemMessage(content=SUPERVISOR_SYSTEM + context)
        messages = [system] + list(state["messages"])
        decision: RouteDecision = router.invoke(messages)
        return {"next": decision.next}

    return supervisor_node


# Verify supervisor with mock — test all three routing outcomes
mock_router = MagicMock()
mock_llm = MagicMock()
mock_llm.with_structured_output.return_value = mock_router

sup_node = make_supervisor(mock_llm)

# Case 1: empty scratchpad → researcher
mock_router.invoke.return_value = RouteDecision(next="researcher")
r1 = sup_node({"messages": [HumanMessage(content="Research quantum computing")],
               "next": "", "scratchpad": None, "report": None, "retry_count": 0})
print(f"Case 1 (empty scratchpad) → {r1['next']}")

# Case 2: scratchpad set → summarizer
mock_router.invoke.return_value = RouteDecision(next="summarizer")
r2 = sup_node({"messages": [HumanMessage(content="Research quantum computing")],
               "next": "", "scratchpad": "Key facts found...", "report": None, "retry_count": 0})
print(f"Case 2 (scratchpad set)   → {r2['next']}")

# Case 3: report set → FINISH
mock_router.invoke.return_value = RouteDecision(next="FINISH")
r3 = sup_node({"messages": [HumanMessage(content="Research quantum computing")],
               "next": "", "scratchpad": "notes...", "report": "# Report...", "retry_count": 0})
print(f"Case 3 (report set)       → {r3['next']}")


**What just happened?**
- The supervisor's system prompt encodes the **routing policy** as natural language rules. The LLM reads the current state context appended to the system prompt and maps it to one of three decisions.
- We tested all three routing cases with mocks — researcher (empty scratchpad), summarizer (scratchpad set), and FINISH (report set). Each test confirms the system prompt semantics are correct.
- Appending the current state as a text snippet to the system prompt is more reliable than expecting the LLM to infer state from the message history alone.


## Step 4 · Research Subagent with DuckDuckGo Search

The research agent uses DuckDuckGo (`duckduckgo-search` package) as a free, no-API-key search tool. In production, swap for Tavily or Google Search — the agent interface is identical.

The researcher's private namespace field is `scratchpad`. It accumulates notes across multiple invocations without overwriting previous findings.


In [ ]:
from duckduckgo_search import DDGS

def search_web(query: str, max_results: int = 5) -> list[dict]:
    """Search DuckDuckGo and return a list of {title, href, body} dicts."""
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=max_results))
        return results
    except Exception as e:
        print(f"  [Search] Error: {e}")
        return []


def format_search_results(results: list[dict]) -> str:
    """Format search results into a readable scratchpad entry."""
    if not results:
        return ""
    lines = []
    for i, r in enumerate(results, 1):
        lines.append(f"[{i}] {r.get('title', 'No title')}")
        lines.append(f"    URL: {r.get('href', '')}")
        lines.append(f"    {r.get('body', '')[:200]}")
        lines.append("")
    return "\n".join(lines)


def researcher_node(state: ResearchState, config: RunnableConfig) -> dict:
    """Research subagent — searches the web and appends to scratchpad."""
    # Extract the research question from the first human message
    question = next(
        (m.content for m in state["messages"] if isinstance(m, HumanMessage)),
        "general research"
    )

    print(f"  [researcher] Searching: {question[:60]}…")
    results = search_web(question, max_results=4)

    if not results:
        # No results: increment retry counter and signal empty
        current_retry = state.get("retry_count", 0)
        return {
            "scratchpad": state.get("scratchpad") or "",  # preserve existing notes
            "retry_count": current_retry + 1,
            "messages": [AIMessage(
                content=f"[Researcher] Search returned no results (attempt {current_retry+1}).",
                name="researcher"
            )],
        }

    # Format and accumulate new notes into scratchpad
    new_notes = format_search_results(results)
    existing = state.get("scratchpad") or ""
    combined = (existing + "\n\n" + new_notes).strip() if existing else new_notes

    print(f"  [researcher] Found {len(results)} results, scratchpad: {len(combined)} chars")
    return {
        "scratchpad": combined,
        "retry_count": 0,  # reset on success
        "messages": [AIMessage(
            content=f"[Researcher] Found {len(results)} sources. Scratchpad updated ({len(combined)} chars).",
            name="researcher"
        )],
    }


# Quick test with a real DuckDuckGo search
print("Testing DuckDuckGo search...")
test_results = search_web("LangGraph multi-agent tutorial", max_results=2)
if test_results:
    print(f"✓ Search works — got {len(test_results)} results")
    print(f"  First result: {test_results[0].get('title', 'N/A')[:60]}")
else:
    print("⚠ Search returned no results (network issue or rate limit — agent will retry)")


**What just happened?**
- **`DDGS().text(query)`** is the synchronous DuckDuckGo search API — free, no API key, but rate-limited. In production use Tavily (`langchain-community TavilySearchResults`) for reliability.
- The scratchpad **accumulates** across researcher invocations — existing notes are preserved and new results are appended. This supports multi-hop research: the supervisor can call the researcher twice with different sub-queries.
- `retry_count` is reset to `0` on success, preventing old failure counts from triggering early FINISH on subsequent sub-queries.


## Step 5 · Summarizer Subagent — Structured Markdown Report

The summarizer reads the `scratchpad` and produces a structured Markdown report. The report format is defined in the system prompt — the LLM fills in the content.

**Report format spec:**
```markdown
# Research Report: [Topic]

## Executive Summary
...

## Key Findings
- ...

## Sources
- [Title](URL)

## Conclusion
...
```


In [ ]:
SUMMARIZER_SYSTEM = """\
You are a research summarizer. Given a scratchpad of raw search results, produce a structured Markdown report.

Output format (use exactly these headers):
# Research Report: [topic]

## Executive Summary
[2-3 sentence overview]

## Key Findings
- [finding 1]
- [finding 2]
- [finding 3]

## Sources
- [title](url)

## Conclusion
[1-2 sentence conclusion]

Do not include any text before the # heading or after the Conclusion section."""


def summarizer_node(state: ResearchState, config: RunnableConfig) -> dict:
    """Summarizer subagent — condenses scratchpad into a structured Markdown report."""
    scratchpad = state.get("scratchpad")
    if not scratchpad:
        # Guard: if called with no scratchpad, produce an error report
        report = "# Research Report\n\n**Error:** No research data available."
        return {
            "report": report,
            "messages": [AIMessage(
                content="[Summarizer] No scratchpad data — produced error report.",
                name="summarizer"
            )],
        }

    # Get the original question for the report title
    question = next(
        (m.content for m in state["messages"] if isinstance(m, HumanMessage)),
        "Research Topic"
    )

    print(f"  [summarizer] Condensing {len(scratchpad)} chars of scratchpad…")

    # Get LLM from config if available (injected via RunnableConfig)
    llm = config.get("configurable", {}).get("llm")

    if llm:
        # Real LLM path
        messages = [
            SystemMessage(content=SUMMARIZER_SYSTEM),
            HumanMessage(content=f"Question: {question}\n\nScratchpad:\n{scratchpad}")
        ]
        response = llm.invoke(messages)
        report = response.content
    else:
        # Simulation path (no LLM key)
        report = f"""# Research Report: {question[:60]}

## Executive Summary
This report summarizes research findings on the topic of {question[:40]}.
The research gathered {scratchpad.count('[') } sources from the web.

## Key Findings
- Finding 1 derived from scratchpad data
- Finding 2 derived from scratchpad data
- Finding 3 derived from scratchpad data

## Sources
- [Simulated source](https://example.com)

## Conclusion
The research is complete. Consult the sources above for detailed information."""

    print(f"  [summarizer] Report generated: {len(report)} chars")
    return {
        "report": report,
        "messages": [AIMessage(
            content=f"[Summarizer] Report complete ({len(report)} chars).",
            name="summarizer"
        )],
    }


# Verify the summarizer guard (no scratchpad)
guard_result = summarizer_node(
    {"messages": [HumanMessage(content="test")],
     "next": "", "scratchpad": None, "report": None, "retry_count": 0},
    config={}
)
print(f"Guard case: report starts with '{guard_result['report'][:30]}'")

# Verify normal path
normal_result = summarizer_node(
    {"messages": [HumanMessage(content="What is quantum computing?")],
     "next": "", "scratchpad": "[1] Quantum basics\n    URL: https://example.com\n    Quantum computers use qubits.",
     "report": None, "retry_count": 0},
    config={}
)
print(f"Normal case: report starts with '{normal_result['report'][:50]}'")


**What just happened?**
- The summarizer accepts the LLM via `config["configurable"]["llm"]` — this makes it testable without a real API key and lets callers inject different models without changing the node code.
- The guard clause produces a named error report (`**Error:** No research data available.`) rather than raising — graphs should never raise exceptions inside nodes; they should encode failures in state.
- The simulation path mirrors the real LLM path's output structure, so the rest of the system (including the capstone challenge) works without an API key.


## Step 6 · Full Graph Assembly with SQLite Checkpointer


In [ ]:
def build_research_graph(llm, checkpointer):
    """Assemble the complete multi-agent research system."""
    supervisor = make_supervisor(llm)

    builder = StateGraph(ResearchState)

    # Add nodes
    builder.add_node("supervisor",  supervisor)
    builder.add_node("researcher",  researcher_node)
    builder.add_node("summarizer",  summarizer_node)

    # Conditional router from supervisor
    builder.add_conditional_edges(
        "supervisor",
        lambda state: state["next"],
        {
            "researcher": "researcher",
            "summarizer": "summarizer",
            "FINISH":     END,
        }
    )

    # All subagents loop back to supervisor
    builder.add_edge("researcher", "supervisor")
    builder.add_edge("summarizer", "supervisor")

    # Entry point
    builder.set_entry_point("supervisor")

    return builder.compile(checkpointer=checkpointer)


# Build with mock LLM (no API key needed for topology verification)
mock_router_main = MagicMock()
mock_llm_main = MagicMock()
mock_llm_main.with_structured_output.return_value = mock_router_main

# Simulate: researcher → summarizer → FINISH (minimum 2 handoffs)
mock_router_main.invoke.side_effect = [
    RouteDecision(next="researcher"),
    RouteDecision(next="summarizer"),
    RouteDecision(next="FINISH"),
]

research_graph = build_research_graph(mock_llm_main, checkpointer)
print("Research graph compiled with SQLite checkpointer")
print("Nodes:", list(research_graph.get_graph().nodes.keys()))


**What just happened?**
- The graph is assembled in the same pattern as Day 7 — supervisor → conditional edges → subagents → back to supervisor. The only additions are `SQLiteSaver` at compile time and the `summarizer` node in place of `writer`.
- The mock simulates the minimum required 2 handoffs (researcher → summarizer → FINISH). The next step will run a full end-to-end test.
- **`compile(checkpointer=checkpointer)`** is the single line that activates persistence — everything else is identical to a non-persistent graph.


## Step 7 · Node-Level Streaming Terminal UI


In [ ]:
async def run_with_streaming(graph, question: str, thread_id: str, llm=None):
    """Run the research graph with node-level and token-level streaming."""
    config = {
        "configurable": {
            "thread_id": thread_id,
            "llm": llm,  # passed through to summarizer_node
        }
    }
    initial_state: ResearchState = {
        "messages": [HumanMessage(content=question)],
        "next": "",
        "scratchpad": None,
        "report": None,
        "retry_count": 0,
    }

    print(f"\n{'='*60}")
    print(f"Research question: {question}")
    print(f"Thread ID: {thread_id}")
    print(f"{'='*60}\n")

    current_node = None
    step_count = 0

    # Combine node-level and token-level streaming
    async for event_type, payload in graph.astream(
        initial_state,
        config=config,
        stream_mode=["updates", "messages"],
    ):
        if event_type == "updates":
            # Node-level: print which node just completed
            for node_name, delta in payload.items():
                if node_name not in ("__start__",):
                    step_count += 1
                    print(f"\n▶ Step {step_count}: [{node_name}]", flush=True)
                    current_node = node_name

                    # Show routing decisions from supervisor
                    if node_name == "supervisor" and "next" in delta:
                        print(f"  → routing to: {delta['next']}", flush=True)

                    # Show scratchpad size when researcher updates it
                    if node_name == "researcher" and delta.get("scratchpad"):
                        print(f"  → scratchpad: {len(delta['scratchpad'])} chars", flush=True)

        elif event_type == "messages":
            # Token-level: stream agent message content inline
            from langchain_core.messages import AIMessageChunk
            chunk, metadata = payload
            node = metadata.get("langgraph_node", "")
            if isinstance(chunk, AIMessageChunk) and chunk.content:
                print(chunk.content, end="", flush=True)

    print(f"\n\n{'='*60}")
    print(f"Completed in {step_count} steps")
    return graph.get_state(config).values


print("Streaming runner defined: run_with_streaming(graph, question, thread_id, llm)")
print("Combines stream_mode=['updates', 'messages'] for a rich terminal UI")


**What just happened?**
- The streaming runner is a thin wrapper around `graph.astream` with both modes active. It doesn't contain any business logic — just event routing and display.
- **`step_count`** gives the user a clear indication of how many agent steps the research took — useful for multi-hop questions that require 3+ handoffs.
- **`graph.get_state(config).values`** at the end retrieves the final persisted state from SQLite — the report is in `result["report"]`.


## Step 8 · End-to-End Test with Multi-Hop Research Question


In [ ]:
import asyncio

async def run_full_system_test():
    """Test the complete system with a mock LLM — 3 handoffs."""
    # Mock: researcher → researcher (second hop) → summarizer → FINISH
    mock_r = MagicMock()
    mock_l = MagicMock()
    mock_l.with_structured_output.return_value = mock_r
    mock_r.invoke.side_effect = [
        RouteDecision(next="researcher"),   # handoff 1
        RouteDecision(next="researcher"),   # handoff 2 (second research hop)
        RouteDecision(next="summarizer"),   # handoff 3
        RouteDecision(next="FINISH"),       # done
    ]

    # Fresh SQLite for this test
    test_checkpointer = SqliteSaver.from_conn_string("/tmp/test_research.db")
    test_graph = build_research_graph(mock_l, test_checkpointer)

    question = "What are the latest breakthroughs in quantum error correction?"
    thread_id = "test-multi-hop-001"

    final_state = await run_with_streaming(test_graph, question, thread_id)

    # Verify the final state
    print("\nFinal state verification:")
    print(f"  messages: {len(final_state['messages'])} total")
    print(f"  scratchpad: {len(final_state.get('scratchpad') or '')} chars")
    print(f"  report: {len(final_state.get('report') or '')} chars")
    print(f"  retry_count: {final_state.get('retry_count', 0)}")

    if final_state.get("report"):
        print("\nReport preview (first 200 chars):")
        print(final_state["report"][:200])

    # Verify persistence: retrieve state by thread_id
    config = {"configurable": {"thread_id": thread_id}}
    persisted = test_graph.get_state(config)
    print(f"\n✓ Persistence verified: state retrieved for thread_id '{thread_id}'")
    print(f"  Persisted report: {'[set]' if persisted.values.get('report') else '[empty]'}")


# Run the test
if asyncio.get_event_loop().is_running():
    await run_full_system_test()
else:
    asyncio.run(run_full_system_test())


**What just happened?**
- The mock simulates **3 handoffs** (researcher × 2, summarizer × 1) — satisfying the capstone requirement.
- The test graph uses a separate SQLite file (`/tmp/test_research.db`) to avoid polluting the production database during development.
- Final verification reads back from SQLite via `get_state` — confirming persistence is active and the report is stored, not just held in memory.


In [ ]:
# Challenge: Build and run the complete capstone research system end-to-end
#
# Implement the full multi-agent research system described in the capstone spec:
#   "Build a multi-agent research system in LangGraph with a supervisor node that
#    routes between a web-search agent and a summarization agent, persists conversation
#    state to a SQLite checkpointer, streams node-level output to a terminal UI, and
#    produces a structured Markdown research report."
#
# All components are implemented above. Wire them together and run:
#
# TASK 1: Run with a real OpenAI key and a live multi-hop research question.
#   llm = ChatOpenAI(model="gpt-4o-mini", streaming=True)
#   checkpointer = SqliteSaver.from_conn_string("/tmp/capstone_research.db")
#   graph = build_research_graph(llm, checkpointer)
#
# TASK 2: Run with this question (requires at least 3 handoffs):
#   question = "Research the history of quantum computing, including key milestones "
#              "since 1980, current leaders in the field, and recent error-correction "
#              "breakthroughs from 2023-2024."
#
# TASK 3: After the run completes, retrieve the report from SQLite by thread_id
#   and print the full Markdown output.
#
# TASK 4: Run a second question in a NEW thread_id, then compare:
#   - The first thread's state (should be unchanged)
#   - The second thread's state (should have its own scratchpad and report)
#
# TASK 5: Verify the output:
#   [ ] streaming printed node names before each agent's tokens
#   [ ] report contains all 4 Markdown sections (Executive Summary, Key Findings, Sources, Conclusion)
#   [ ] SQLite persisted both threads independently
#   [ ] total handoffs >= 3
#
# Scaffold:
# async def capstone_run():
#     llm = ChatOpenAI(model="gpt-4o-mini", streaming=True)  # set OPENAI_API_KEY
#     checkpointer = SqliteSaver.from_conn_string("/tmp/capstone_research.db")
#     graph = build_research_graph(llm, checkpointer)
#
#     question = "Research the history of quantum computing..."
#     final_state = await run_with_streaming(graph, question, "capstone-001", llm=llm)
#
#     # Print full report
#     print("\n" + "="*60)
#     print("FINAL RESEARCH REPORT")
#     print("="*60)
#     print(final_state.get("report", "[No report generated]"))
#
# await capstone_run()

# Your solution here


---
## Day 10 key concepts recap

| Concept | What to remember |
|---|---|
| Architecture-first | Define state schema and wire checkpointer before writing agent logic |
| `SqliteSaver` | `from_conn_string(path)` — persists across process restarts; `get_state(config)` retrieves |
| Supervisor routing | Appends current state as text to system prompt so the LLM can make data-aware decisions |
| DuckDuckGo search | `DDGS().text(query, max_results=N)` — free, no key; swap for Tavily in production |
| Scratchpad accumulation | Append new research notes to existing scratchpad; don't overwrite |
| Summarizer via config | Pass LLM through `config["configurable"]["llm"]` for testability |
| Dual-mode streaming | `stream_mode=["updates", "messages"]` → `(event_type, payload)` tuples |
| Multi-hop verification | Count handoffs in trace; confirm ≥ 3 for a complex research question |

> **Tip:** The capstone is an integration test for everything you have built this week. Start with the state schema and wiring before writing any prompt — the graph structure is harder to change than the prompts.

---
## What's next
**Congratulations — you have completed the Building AI Agents with LangGraph course!**

Next steps:
- Deploy your research system to LangGraph Platform (`langgraph build` + `langgraph deploy`)
- Replace `InMemoryStore` with a persistent vector store for semantic memory search
- Add a Human-in-the-Loop node so a user can approve the research plan before execution begins
- Wire Tavily search for production-grade web search reliability

Mark Day 10 complete in your [tracker](../index.html).
